# Week 3 Exercise – Synthetic Data Generator (winniekariuki)

Generate structured **synthetic datasets** from a natural-language description. Uses **OpenAI** (GPT-4o-mini) and **Llama 3.2** (Ollama) with multiple prompt templates for diverse outputs, and a **Gradio UI**.

In [11]:
# imports
import os
import re
import json
import io
import csv
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [12]:
# environment & API clients (same pattern as Week 2)
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai_client = OpenAI() if (api_key and api_key.startswith("sk-") and len(api_key) > 10) else None
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

if openai_client:
    print("OpenAI client ready (GPT).")
else:
    print("OPENAI_API_KEY not set — add to .env to use GPT.")
print("Ollama client ready (Llama). Ensure Ollama is running: ollama run llama3.2")

OpenAI client ready (GPT).
Ollama client ready (Llama). Ensure Ollama is running: ollama run llama3.2


## Prompt templates for diverse outputs

Different schemas and instructions so the same scenario can yield varied synthetic data (reviews, products, tickets, jobs, surveys, or custom).

In [13]:
# Prompt templates: name -> (system_snippet, user_instruction_snippet)
# System prompt is shared; user prompt is built with scenario + num_rows + template-specific guidance.
PROMPT_TEMPLATES = {
    "Customer reviews": (
        "Each record must have: reviewer_name (fake), rating (1-5), review_text (1-2 sentences), date (YYYY-MM-DD), product or service name.",
        "Generate realistic but fake customer reviews. Mix positive and negative; vary length and tone."
    ),
    "Product catalog": (
        "Each record must have: name, price (number), category, description (short), sku (alphanumeric).",
        "Generate a product catalog. Use diverse categories and realistic-looking but fake product names and SKUs."
    ),
    "Support tickets": (
        "Each record must have: ticket_id, subject, status (open/in_progress/resolved), priority (low/medium/high), created_at (ISO date), customer_segment.",
        "Generate support ticket entries. Vary status and priority; use short, realistic subject lines."
    ),
    "Job postings": (
        "Each record must have: title, company (fake name), location, salary_range (e.g. 80k-120k), description (1-2 sentences), requirements (short list or string).",
        "Generate job postings. Use varied roles, locations, and salary ranges; all data must be synthetic."
    ),
    "Survey responses": (
        "Each record must have: respondent_id, question_id, answer_text, score (1-5 or N/A), timestamp (ISO).",
        "Generate survey response rows. Vary answers and scores; keep timestamps realistic."
    ),
    "Custom": (
        "Infer a sensible schema from the user's scenario. Each record must be a JSON object with consistent keys across all records.",
        "Generate synthetic data that matches the user's description. No real PII; use plausible fake values."
    ),
}

SYSTEM_PROMPT_BASE = """You are a synthetic dataset generator. Output ONLY valid data — no explanations, no markdown code fences, no extra text.
- Generate realistic but fake data. No real names, emails, or identifiable information.
- For JSON: output a single JSON array of objects. Each object is one record with consistent keys.
- Generate exactly the number of records requested."""

In [14]:
def build_prompt(scenario: str, num_rows: int, template_name: str):
    """Build system and user messages for the chosen template."""
    template = PROMPT_TEMPLATES.get(template_name, PROMPT_TEMPLATES["Custom"])
    schema_hint, diversity_hint = template
    system = SYSTEM_PROMPT_BASE + "\n" + schema_hint
    user = (
        f"Generate a synthetic dataset with exactly {num_rows} records. "
        f"Scenario: {scenario.strip() or 'general business data'}. "
        f"{diversity_hint} "
        "Output a single JSON array of objects only. No other text."
    )
    return system, user

In [15]:
def generate_with_openai(system: str, user: str) -> str:
    """Call OpenAI; return raw JSON string."""
    if not openai_client:
        return json.dumps({"error": "OpenAI client not configured. Set OPENAI_API_KEY in .env."})
    try:
        r = openai_client.chat.completions.create(
            model=MODEL_GPT,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            response_format={"type": "json_object"},
            temperature=0.7,
        )
        raw = (r.choices[0].message.content or "").strip()
        # response_format json_object may return {"key": [...]}; normalize to array for display
        data = json.loads(raw)
        if isinstance(data, dict) and len(data) == 1 and isinstance(list(data.values())[0], list):
            data = list(data.values())[0]
        if isinstance(data, list):
            raw = json.dumps(data, indent=2)
        return raw
    except Exception as e:
        return json.dumps({"error": str(e)})


def generate_with_ollama(system: str, user: str) -> str:
    """Call Ollama (Llama); return raw JSON string. Strip markdown if present."""
    try:
        r = ollama_client.chat.completions.create(
            model=MODEL_LLAMA,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0.7,
        )
        raw = (r.choices[0].message.content or "").strip()
        # Remove markdown code blocks if present
        if raw.startswith("```"):
            raw = re.sub(r"^```\w*\n", "", raw)
            raw = re.sub(r"\n```\s*$", "", raw)
        # Normalize to array for display
        data = json.loads(raw)
        if isinstance(data, dict) and len(data) == 1 and isinstance(list(data.values())[0], list):
            data = list(data.values())[0]
        if isinstance(data, list):
            raw = json.dumps(data, indent=2)
        return raw
    except json.JSONDecodeError:
        return json.dumps({"error": "Model did not return valid JSON. Try a shorter scenario or fewer rows."})
    except Exception as e:
        return json.dumps({"error": str(e)})

In [16]:
def json_to_csv(json_str: str) -> str:
    """Convert JSON array of objects to CSV string."""
    try:
        data = json.loads(json_str)
        if isinstance(data, dict) and "error" in data:
            return json_str
        if not isinstance(data, list):
            data = [data]
        if not data:
            return ""
        out = io.StringIO()
        writer = csv.DictWriter(out, fieldnames=list(data[0].keys()))
        writer.writeheader()
        writer.writerows(data)
        return out.getvalue()
    except Exception:
        return json_str


def generate_synthetic_data(scenario: str, num_rows: int, output_format: str, template_name: str, model_label: str) -> str:
    """Generate dataset and return CSV or JSON string."""
    num_rows = max(1, min(int(num_rows), 50))
    system, user = build_prompt(scenario or "miscellaneous records", num_rows, template_name)

    if model_label == "GPT-4o-mini (OpenAI)":
        raw = generate_with_openai(system, user)
    elif model_label == "Llama 3.2 (Ollama)":
        raw = generate_with_ollama(system, user)
    else:
        raw = json.dumps({"error": "Unknown model."})

    if output_format == "CSV":
        return json_to_csv(raw)
    return raw

## Gradio UI

Scenario, row count, output format, prompt template, and model selector. Output is raw CSV or JSON for copy or download.

In [17]:
with gr.Blocks(theme=gr.themes.Soft(), title="Synthetic Data Generator – Week 3 (winniekariuki)") as demo:
    gr.Markdown(
        "# Synthetic Data Generator\n"
        "Describe the kind of data you want (e.g. *restaurant reviews with rating and date*). "
        "Pick a **prompt template** for schema diversity, choose **OpenAI** or **Llama**, and generate. Output as CSV or JSON."
    )
    with gr.Row():
        scenario = gr.Textbox(
            label="Dataset scenario",
            placeholder="e.g. Restaurant reviews with reviewer_name, rating (1-5), review_text, date",
            lines=2,
        )
    with gr.Row():
        num_rows = gr.Slider(1, 50, value=5, step=1, label="Number of rows")
        output_format = gr.Dropdown(["JSON", "CSV"], value="JSON", label="Output format")
        template_name = gr.Dropdown(
            list(PROMPT_TEMPLATES.keys()),
            value="Customer reviews",
            label="Prompt template",
        )
        model_choice = gr.Radio(
            ["GPT-4o-mini (OpenAI)", "Llama 3.2 (Ollama)"],
            label="Model",
            value="GPT-4o-mini (OpenAI)",
        )
    btn = gr.Button("Generate", variant="primary")
    out = gr.Textbox(label="Generated data", lines=14)

    btn.click(
        fn=generate_synthetic_data,
        inputs=[scenario, num_rows, output_format, template_name, model_choice],
        outputs=out,
    )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
